## Common imports

In [1]:
import matplotlib.pyplot as plt
plt.rc("axes.spines", top=False, right=False)

import seaborn as sns
import pandas as pd
import numpy as np
from scipy import stats

from tqdm import tqdm

plot_path="../output/plots"
mypath = plot_path

import sys
import os
sys.path.append("../src")
from Branch_analysis import branch_dsc, update_DSCs, update_compartments
from Summary_analysis import summary_dsc, summary_compt, avgCDF
from plots import plot_dsc

In [ ]:
from pathlib import Path

root = Path('../repo_data/')
if not root.exists():
    raise FileNotFoundError(f"Root folder '{root}' not found.")

## import and analyse ExM

In [3]:
# Initialize the ExM data dictionary
ExMdata = {}

# Helper function to add dendrites
def add_dendrite(data, dendrite_name, subfolder, filename, ex_factor, res):
    folder_path = root / subfolder / 'tables'
    data[dendrite_name] = {
        'path': folder_path / filename,
        'file': filename,
        'ex_factor': ex_factor,
        'res': res
    }

# Add dendrites with helper function
add_dendrite(ExMdata, 'dendrite1', 'ExM/2016_07_07/branch1', '2016_07_07_slice4_left_branch1_Local Diameter.csv', 4.1, 1)
# Uncomment if needed after verification
# add_dendrite(ExMdata, 'dendrite2', 'ExM/2016_07_07/branch2', '2016_07_07_slice4_left_branch2_Local Diameter.csv', 4.1, 1)
add_dendrite(ExMdata, 'dendrite3', 'ExM/2016_07_07/branch3', '2016_07_07_slice4_left_branch3_Local Diameter.csv', 4.1, 1)
add_dendrite(ExMdata, 'dendrite4', 'ExM/2019_05_04/branch4', '2019_05_04_slice3_side1_branch4_localDiameter.csv', 4.1, 1)

add_dendrite(ExMdata, 'C1_02_cell1_d2', 'ExM/2020_06_08/T1_02', '20210201_DG-T1_AF488-02_neuron01_8bit_1000-3000_dend2 - Local Diameter.csv', 3.8, 0.173)
add_dendrite(ExMdata, 'C1_03_cell2_d2', 'ExM/2020_06_08/T1_03', '20210201_dg-t1_af488-03_decon-Resolution Level 1_8bit_480-2600_dend2 - Local Diameter.csv', 3.8, 0.173)
add_dendrite(ExMdata, 'C1_03_cell2_d3', 'ExM/2020_06_08/T1_03', '20210201_dg-t1_af488-03_decon-Resolution Level 1_8bit_480-2600_dend3 - Local Diameter.csv', 3.8, 0.173)

# Optionally, print or log ExMdata to verify paths and parameters
print(ExMdata)

{'dendrite1': {'path': WindowsPath('F:/bonn/data/Distal_dendrite/morphology/repo_data/ExM/2016_07_07/branch1/tables/2016_07_07_slice4_left_branch1_Local Diameter.csv'), 'file': '2016_07_07_slice4_left_branch1_Local Diameter.csv', 'ex_factor': 4.1, 'res': 1}, 'dendrite3': {'path': WindowsPath('F:/bonn/data/Distal_dendrite/morphology/repo_data/ExM/2016_07_07/branch3/tables/2016_07_07_slice4_left_branch3_Local Diameter.csv'), 'file': '2016_07_07_slice4_left_branch3_Local Diameter.csv', 'ex_factor': 4.1, 'res': 1}, 'dendrite4': {'path': WindowsPath('F:/bonn/data/Distal_dendrite/morphology/repo_data/ExM/2019_05_04/branch4/tables/2019_05_04_slice3_side1_branch4_localDiameter.csv'), 'file': '2019_05_04_slice3_side1_branch4_localDiameter.csv', 'ex_factor': 4.1, 'res': 1}, 'C1_02_cell1_d2': {'path': WindowsPath('F:/bonn/data/Distal_dendrite/morphology/repo_data/ExM/2020_06_08/T1_02/tables/20210201_DG-T1_AF488-02_neuron01_8bit_1000-3000_dend2 - Local Diameter.csv'), 'file': '20210201_DG-T1_AF488

In [4]:
dataSet=ExMdata
for name, dendrite in tqdm(dataSet.items()):
    #print(name)
    #print(data[name]['path'])
    
    file_path = dendrite['path']#+dendrite['file']
    dataSet[name] = branch_dsc(file_path, dataSet[name]['res'], dataSet[name]['ex_factor'])

100%|██████████| 6/6 [00:00<00:00, 12.05it/s]


manually adjust

In [5]:
dendrite = ExMdata['C1_03_cell2_d3']
DSC=dendrite['data'][3947:4047].copy() # makes a copy to work on
DSC.diam*=1/dendrite['ex_factor']   #if the DSCs overlap each other the scaling applied again. Separate DSC from orig data
DSC.smth*=1/dendrite['ex_factor']
DSC.length*=dendrite['res']/dendrite['ex_factor']
DSC.pt_distance*=dendrite['res']/dendrite['ex_factor']
dendrite['DSCs'].append(DSC) # add the changes back to the copy of the original data

update_DSCs(dendrite)
update_compartments(dendrite)

dendrite = ExMdata['C1_03_cell2_d2']
del dendrite['DSCs'][2] #the auto detected DSC is too long past the DSC -shortened

update_DSCs(dendrite)
update_compartments(dendrite)


dendrite = ExMdata['dendrite1']
del dendrite['DSCs'][11] #noisey false positives
del dendrite['DSCs'][10] #noisey false positives
del dendrite['DSCs'][9] #noisey false positives
del dendrite['DSCs'][8] #noisey false positives
del dendrite['DSCs'][7] #
del dendrite['DSCs'][2] #false positive

update_DSCs(dendrite)
update_compartments(dendrite)

In [6]:
#select property to plot [mean, lenght, distance]
parameter='mean'
ExM_df= summary_dsc(ExMdata, parameter)

## Import and analyse STED

In [7]:
# Define paths for different datasets
paths = {
    '3wk': {'path': root / 'STED/3week_thy1_dg/tables/'},
    '5wk': {'path': root / 'STED/5week_thy1_dg/tables/'},
    #'prox': {'path': root / 'STED/dg_proximal/tables/'}
}

# Initialize STEDdata dictionary
STEDdata = {}

# Loop through each dataset
for dataset, d in paths.items():
    path_to_data = d['path']
    STEDdata[dataset] = {}
    
    # Get files matching the pattern in the dataset path
    for file in path_to_data.glob("*Local*"):
        print(file)  # Optional: print to confirm file paths
        name = file.name
        dendrite = 'dendrite' + name.split(' - ')[0]
        
        # Populate STEDdata with file details
        STEDdata[dataset][dendrite] = {
            'path': file,
            'file': name,
            'ex_factor': 1,
            'res': 0.04
        }

# Optionally, print or log STEDdata to verify paths and parameters
print(STEDdata)

F:\bonn\data\Distal_dendrite\morphology\repo_data\STED\3week_thy1_dg\tables\16 - Local Diameter.csv
F:\bonn\data\Distal_dendrite\morphology\repo_data\STED\3week_thy1_dg\tables\17 - Local Diameter.csv
F:\bonn\data\Distal_dendrite\morphology\repo_data\STED\3week_thy1_dg\tables\21 - Local Diameter.csv
F:\bonn\data\Distal_dendrite\morphology\repo_data\STED\3week_thy1_dg\tables\25 - Local Diameter.csv
F:\bonn\data\Distal_dendrite\morphology\repo_data\STED\3week_thy1_dg\tables\25-2 - Local Diameter.csv
F:\bonn\data\Distal_dendrite\morphology\repo_data\STED\3week_thy1_dg\tables\27 - Local Diameter.csv
F:\bonn\data\Distal_dendrite\morphology\repo_data\STED\3week_thy1_dg\tables\28 - Local Diameter.csv
F:\bonn\data\Distal_dendrite\morphology\repo_data\STED\3week_thy1_dg\tables\29 - Local Diameter.csv
F:\bonn\data\Distal_dendrite\morphology\repo_data\STED\3week_thy1_dg\tables\29-4 - Local Diameter.csv
F:\bonn\data\Distal_dendrite\morphology\repo_data\STED\3week_thy1_dg\tables\30 - Local Diameter.

In [8]:
for n, dataSet in STEDdata.items():
    print(n)
    
    for name, dendrite in tqdm(dataSet.items()):
        #print(name)
        #print(data[name]['path'])
        
        file_path = dendrite['path']#+dendrite['file']
        dataSet[name] = branch_dsc(file_path, dataSet[name]['res'], dataSet[name]['ex_factor'])

3wk


100%|██████████| 29/29 [00:00<00:00, 138.50it/s]


5wk


100%|██████████| 21/21 [00:00<00:00, 202.25it/s]


manually adjust

In [9]:
# Helper function to delete DSCs and update
def delete_DSCs_and_update(branch, indices):
    for idx in sorted(indices, reverse=True):  # Delete from end to avoid index shifting
        del branch['DSCs'][idx]
    update_DSCs(branch)
    update_compartments(branch)

# Helper function to update DSC properties
def update_DSC_properties(branch, prop_idx, mean_range, length_start, length_end):
    DSC = branch['DSCs'][prop_idx]
    branch['DSC_properties'][prop_idx]['mean'] = DSC.smth[mean_range[0]:mean_range[1]].mean()
    branch['DSC_properties'][prop_idx]['length'] = DSC.length[DSC.length.index[0] + length_end] - DSC.length[DSC.length.index[0] + length_start]

# Helper function to add a new DSC and update
def add_DSC_and_update(branch, data_range):
    DSC = branch['data'][data_range[0]:data_range[1]].copy()
    DSC.diam *= 1 / branch['ex_factor']
    DSC.smth *= 1 / branch['ex_factor']
    DSC.length *= branch['res'] / branch['ex_factor']
    DSC.pt_distance *= branch['res'] / branch['ex_factor']
    branch['DSCs'].append(DSC)
    update_DSCs(branch)
    update_compartments(branch)

# 3wks modifications
delete_DSCs_and_update(STEDdata['3wk']['dendrite25'], [0, 1])
delete_DSCs_and_update(STEDdata['3wk']['dendrite28'], [0, 1])
delete_DSCs_and_update(STEDdata['3wk']['dendrite29-4'], [0])
delete_DSCs_and_update(STEDdata['3wk']['dendrite39-21'], [2])

update_DSC_properties(STEDdata['3wk']['dendrite40-4'], 0, (37, 47), 37, 47)
delete_DSCs_and_update(STEDdata['3wk']['dendrite41-0'], [1])

# 5wks modifications
add_DSC_and_update(STEDdata['5wk']['dendrite36-2'], (145, 270))
update_DSC_properties(STEDdata['5wk']['dendrite38-1'], 0, (11, 32), 11, 32)
add_DSC_and_update(STEDdata['5wk']['dendrite42-2'], (50, 150))

delete_DSCs_and_update(STEDdata['5wk']['dendrite14'], [3])
delete_DSCs_and_update(STEDdata['5wk']['dendrite43-1'], [0])
delete_DSCs_and_update(STEDdata['5wk']['dendrite43-2'], [0])
delete_DSCs_and_update(STEDdata['5wk']['dendrite55'], [0])

data

In [10]:
parameter='mean'

df3= summary_dsc(STEDdata['3wk'], parameter)
df5= summary_dsc(STEDdata['5wk'], parameter)

STED_df = pd.concat([df3, df5]).stack().reset_index(drop=True)


## Import and analyse EM

In [11]:
# Helper function to add branches
def add_branch(data, branch_name, folder, filename, ex_factor=1, res=0.01):
    data[branch_name] = {
        'path': os.path.join(folder, filename),
        'ex_factor': ex_factor,
        'res': res
    }

# Initialize the EM data dictionary
EMdata = {}

# Set up folders and files
folder1 = os.path.join(root, 'EM', 'DD0075_E5', 'tables')
folder2 = os.path.join(root, 'EM', '1922_DistalMolLayer', 'tables')

# Add branches with helper function
add_branch(EMdata, 'branch1', folder1, 'D1_binary - Local Diameter.csv')
add_branch(EMdata, 'branch2', folder1, 'D2_binary - Local Diameter.csv')
add_branch(EMdata, 'branch4', folder1, 'D4_binary - Local Diameter.csv')
add_branch(EMdata, 'branch5', folder1, 'D5_binary - Local Diameter.csv')

add_branch(EMdata, 'branch3', folder1, '03 - Local Diameter.csv')
add_branch(EMdata, 'branch6', folder1, '06 - Local Diameter.csv')
add_branch(EMdata, 'branch7', folder1, '07 - Local Diameter.csv')
add_branch(EMdata, 'branch8', folder1, '08 - Local Diameter.csv')
add_branch(EMdata, 'branch9', folder1, '09 - Local Diameter.csv')

add_branch(EMdata, 'branch12', folder2, 'object02 - Local Diameter.csv')
add_branch(EMdata, 'branch13', folder2, 'object03 - Local Diameter.csv')
add_branch(EMdata, 'branch14', folder2, 'object04 - Local Diameter.csv')
add_branch(EMdata, 'branch17', folder2, 'object07 - Local Diameter.csv')

# Uncomment if needed after verification with Michael
# add_branch(EMdata, 'branch19', folder2, 'object092 - Local Diameter.csv')

add_branch(EMdata, 'branch20', folder2, 'object10 - Local Diameter.csv')
add_branch(EMdata, 'branch21', folder2, 'object11 - Local Diameter.csv')
add_branch(EMdata, 'branch22', folder2, 'object12 - Local Diameter.csv')
add_branch(EMdata, 'branch25', folder2, 'object15 - Local Diameter.csv')

# Optionally, print or log EMdata to verify paths and parameters
print(EMdata)

{'branch1': {'path': 'F:\\bonn\\data\\Distal_dendrite\\morphology\\repo_data\\EM\\DD0075_E5\\tables\\D1_binary - Local Diameter.csv', 'ex_factor': 1, 'res': 0.01}, 'branch2': {'path': 'F:\\bonn\\data\\Distal_dendrite\\morphology\\repo_data\\EM\\DD0075_E5\\tables\\D2_binary - Local Diameter.csv', 'ex_factor': 1, 'res': 0.01}, 'branch4': {'path': 'F:\\bonn\\data\\Distal_dendrite\\morphology\\repo_data\\EM\\DD0075_E5\\tables\\D4_binary - Local Diameter.csv', 'ex_factor': 1, 'res': 0.01}, 'branch5': {'path': 'F:\\bonn\\data\\Distal_dendrite\\morphology\\repo_data\\EM\\DD0075_E5\\tables\\D5_binary - Local Diameter.csv', 'ex_factor': 1, 'res': 0.01}, 'branch3': {'path': 'F:\\bonn\\data\\Distal_dendrite\\morphology\\repo_data\\EM\\DD0075_E5\\tables\\03 - Local Diameter.csv', 'ex_factor': 1, 'res': 0.01}, 'branch6': {'path': 'F:\\bonn\\data\\Distal_dendrite\\morphology\\repo_data\\EM\\DD0075_E5\\tables\\06 - Local Diameter.csv', 'ex_factor': 1, 'res': 0.01}, 'branch7': {'path': 'F:\\bonn\\data

In [12]:
dataSet=EMdata
for name, dendrite in tqdm(dataSet.items()):
    #print(name)
    #print(data[name]['path'])
    
    file_path = dendrite['path']#+dendrite['file']
    dataSet[name] = branch_dsc(file_path, dataSet[name]['res'], dataSet[name]['ex_factor'], p=100, pdiff=1)

100%|██████████| 17/17 [00:00<00:00, 141.09it/s]


Manually adjust

In [13]:
### add manual to the data dictionary 
#   
branch=EMdata['branch4']
DSC=branch['data'][120:275].copy()
DSC.diam*=1/branch['ex_factor']   
DSC.smth*=1/branch['ex_factor']
DSC.length*=branch['res']/branch['ex_factor']
DSC.pt_distance*=branch['res']/branch['ex_factor']
branch['DSCs'].append(DSC) 

##add manual branch5 goes below x2 threshold but slight increase excludes it from algorithm even though still DSC in my opinion
branch=EMdata['branch5']
DSC=branch['data'][360:700].copy()
DSC.diam*=1/branch['ex_factor']   
DSC.smth*=1/branch['ex_factor']
DSC.length*=branch['res']/branch['ex_factor']
DSC.pt_distance*=branch['res']/branch['ex_factor']
branch['DSCs'].append(DSC)  

branch=EMdata['branch14']
DSC=branch['data'][364:450].copy()
DSC.diam*=1/branch['ex_factor']   
DSC.smth*=1/branch['ex_factor']
DSC.length*=branch['res']/branch['ex_factor']
DSC.pt_distance*=branch['res']/branch['ex_factor']
branch['DSCs'].append(DSC)  


##maybe other DSCs in branch 17 object7 but difficult to confirm without proper registration.
del EMdata['branch17']['DSCs'][1]

update_DSCs(EMdata['branch4'])
update_DSCs(EMdata['branch5'])

# update_DSCs(EMdata['branch8']) # end of dendrite
update_DSCs(EMdata['branch14'])

update_DSCs(EMdata['branch17'])
# update_DSCs(EMdata['branch20'])

In [14]:
#select property to plot [mean, lenght]
parameter='mean'

EM_df= summary_dsc(EMdata, parameter)


## stats

In [15]:
# Prepare 1D numeric arrays (drop NaNs) for ANOVA
em_vals = EM_df.stack().dropna().astype(float).to_numpy()
sted_vals = STED_df.dropna().astype(float).to_numpy()
exm_vals = ExM_df.stack().dropna().astype(float).to_numpy()

# Kruskal Wallis
h_stat, p_val = stats.kruskal(em_vals, sted_vals, exm_vals)

print("Kruskal Wallis (EM vs STED vs ExM)")
print(f"n_EM={len(em_vals)}, n_STED={len(sted_vals)}, n_ExM={len(exm_vals)}")
print(f"h-statistic = {h_stat:.4f}")
print(f"p-value     = {p_val:.4e}")

alpha = 0.05
if p_val < alpha:
    print("Result: Significant difference between at least one group mean.")
else:
    print("Result: No significant difference between group means.")

Kruskal Wallis (EM vs STED vs ExM)
n_EM=9, n_STED=37, n_ExM=35
h-statistic = 5.8593
p-value     = 5.3415e-02
Result: No significant difference between group means.


In [16]:
# Pairwise post-hoc test after Kruskal-Wallis: Mann-Whitney U

pairs = [
	("EM", "STED", em_vals, sted_vals),
	("EM", "ExM", em_vals, exm_vals),
	("STED", "ExM", sted_vals, exm_vals),
]

rows = []
for g1, g2, v1, v2 in pairs:
	u_stat, p_value = stats.mannwhitneyu(v1, v2, alternative="two-sided")
	rows.append(
		{
			"group1": g1,
			"group2": g2,
			"u_stat": u_stat,
			"p_value": p_value,
		}
	)

mannwhit_df = pd.DataFrame(rows)

# Bonferroni correction for 3 pairwise tests
m = len(mannwhit_df)
mannwhit_df["p_adj_bonf"] = np.minimum(mannwhit_df["p_value"] * m, 1.0)
mannwhit_df["reject"] = mannwhit_df["p_adj_bonf"] < alpha

display(mannwhit_df)

,group1,group2,u_stat,p_value,p_adj_bonf,reject
0,EM,STED,97.0,0.056057,0.168170,False
1,EM,ExM,151.0,0.861414,1.000000,False
2,STED,ExM,830.0,0.040313,0.120939,False


In [17]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Post-hoc test after ANOVA: Tukey HSD (which pairs differ)

all_vals = np.concatenate([em_vals, sted_vals, exm_vals])
group_labels = (
	["EM"] * len(em_vals)
	+ ["STED"] * len(sted_vals)
	+ ["ExM"] * len(exm_vals)
)

tukey = pairwise_tukeyhsd(endog=all_vals, groups=group_labels, alpha=alpha)
print(tukey)

# Optional: table format for easier reading
tukey_df = pd.DataFrame(
	tukey._results_table.data[1:],
	columns=tukey._results_table.data[0]
)
display(tukey_df)

 Multiple Comparison of Means - Tukey HSD, FWER=0.05  
group1 group2 meandiff p-adj   lower    upper   reject
------------------------------------------------------
    EM    ExM   8.5078 0.9556 -62.2489  79.2644  False
    EM   STED  56.4721 0.1405 -13.8923 126.8365  False
   ExM   STED  47.9644 0.0323    3.324  92.6047   True
------------------------------------------------------


,group1,group2,meandiff,p-adj,lower,upper,reject
0,EM,ExM,8.5078,0.9556,-62.2489,79.2644,False
1,EM,STED,56.4721,0.1405,-13.8923,126.8365,False
2,ExM,STED,47.9644,0.0323,3.3240,92.6047,True


In [39]:
exm_vals.mean()

np.float64(309.17246526377625)

In [37]:
sted_vals.mean()

np.float64(357.13682565844795)